# 2주차 (b) — 전처리 파이프라인 + Naive·ARIMA 베이스라인

> 계획서 v5.1 §9 2주차 작업 — 산출물 3건(02) 외 나머지 5개 작업

## 본 노트북의 목표

| 작업 | 계획서 근거 | 대응 절 |
|------|------------|--------|
| 1. 전처리 파이프라인 (정책 변수 t-1, Robust Scaler train-fit) | §4.3, CL-02·CL-05 | §3, §7 |
| 2. Lag/Rolling Feature (`shift(k)`, `.rolling().mean().shift(1)`) | §4.3, CL-03 | §5, §6 |
| 3. Naive 베이스라인 (`Δŷ=0`) — RMSE 측정 | §5.1, §9 검증 포인트 | §8 |
| 4. ARIMA 베이스라인 (자기상관 학습 능력) | §5.1 | §9 |
| 5. 누수 코드 리뷰 (체크리스트 7개 자동 검증) | `data_leakage_checklist.md` | §10 |
| 6. 리포트 템플릿 골격 | §8 산출물 | §11 |

## 입력 / 출력

**입력**: `data/processed/features_v1_candidate.csv` (1주차 freeze 9 + 타겟)

**출력**:
- `data/processed/features_with_lags_v1.csv` — Lag/Rolling 적용 + scaled
- `data/processed/baseline_results.csv` — Naive·ARIMA RMSE/MAE/방향성
- `reports/figures/w2_04_baseline_compare.png`
- `reports/report_skeleton.md` — 최종 리포트 골격

## 후속 단계 (3주차 → `03_freeze_xgboost.ipynb`)

변수 freeze 최종 확정·5주차 ablation 대상 명시·XGBoost 분위수 회귀.

---

## 0. 환경 설정

In [ ]:
# === 의존성 자동 체크 ===
import importlib.util, subprocess, sys

REQUIRED = {
    'statsmodels': 'statsmodels',
    'matplotlib':  'matplotlib',
    'sklearn':     'scikit-learn',
    'yaml':        'pyyaml',
}
for _import_name, _pip_name in REQUIRED.items():
    if importlib.util.find_spec(_import_name) is None:
        print(f'  Installing {_pip_name} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _pip_name])
print('✅ 의존성 체크 완료\n')

import re
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

PROJECT_ROOT = Path.cwd().parent
DATA_DIR     = PROJECT_ROOT / 'data'
FIG_DIR      = PROJECT_ROOT / 'reports' / 'figures'
REPORT_DIR   = PROJECT_ROOT / 'reports'
DOCS_DIR     = PROJECT_ROOT / 'docs'
CONFIG_PATH  = PROJECT_ROOT / 'configs' / 'config.yaml'
FIG_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH, encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 180)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'config split: {CONFIG["split"]}')

---

## 1. 1주차 산출물 로드 (9 freeze + 타겟)

In [ ]:
features_v1 = pd.read_csv(
    DATA_DIR / 'processed' / 'features_v1_candidate.csv',
    index_col='date', parse_dates=['date']
).sort_index()

TARGET = CONFIG['project']['target']
FREEZE_FEATURES = [c for c in features_v1.columns if c != TARGET]
POLICY_VARS = [v for v in ['kr_base_rate', 'us_fed_funds'] if v in FREEZE_FEATURES]

print(f'features_v1.shape: {features_v1.shape}')
print(f'기간: {features_v1.index.min().date()} ~ {features_v1.index.max().date()}')
print(f'타겟: {TARGET}')
print(f'freeze 9: {FREEZE_FEATURES}')
print(f'정책 변수 (CL-05 대상): {POLICY_VARS}')

# CL-07 — 한국 휴장일(타겟 결측) 행 drop
n_before = len(features_v1)
features_v1 = features_v1.dropna(subset=[TARGET])
n_after = len(features_v1)
print(f'\nCL-07 한국 휴장일 drop: {n_before} → {n_after}  (-{n_before - n_after}행)')

---

## 2. 데이터 분할 (계획서 §4.2)

| 구간 | 기간 | 용도 |
|------|------|------|
| Train | 2010-01 ~ 2020-12 | LSTM 학습, ARIMA 학습 |
| Calibration | 2021-01 ~ 2021-12 | (선택) Conformal 보정 |
| Validation | 2022-01 ~ 2022-12 | 하이퍼파라미터 튜닝 |
| Test | 2023-01 ~ 2025-12 | 최종 평가 |

> 본 노트북에서 베이스라인 평가는 Val + Test 에서 수행. Cal 은 LSTM 단계 (4-5주차) 에서 사용.

In [ ]:
# 계획서 §4.2 분할 (config.yaml 의 train 끝을 2020-12 로 분할 — Cal 신설)
SPLIT = {
    'train': ('2010-01-01', '2020-12-31'),
    'cal':   ('2021-01-01', '2021-12-31'),
    'val':   ('2022-01-01', '2022-12-31'),
    'test':  ('2023-01-01', '2025-12-31'),
}

def slice_period(df, period):
    s, e = SPLIT[period]
    return df.loc[s:e]

print(f"{'구간':10s}{'기간':28s}{'행 수':>10s}")
for k in SPLIT:
    sub = slice_period(features_v1, k)
    print(f"{k:10s}{SPLIT[k][0]} ~ {SPLIT[k][1]}   {len(sub):>10,d}")

---

## 3. 정책 변수 t-1 시프트 (CL-05)

한은 금통위/FOMC 발표 당일 결과를 input 으로 쓰면 누수. **항상 `t-1` 값 사용** 강제.

In [ ]:
# 정책 변수만 shift(1) — 시장 변수(국고채 종가 등)는 종가가 t 시점에 known 이라 shift 불필요
features_safe = features_v1.copy()
for var in POLICY_VARS:
    features_safe[var] = features_safe[var].shift(1)

# shift 후 첫 행 NaN 처리: 첫 영업일을 drop (CL-08 train 통계 보호)
features_safe = features_safe.dropna(subset=POLICY_VARS)

print(f'CL-05 적용 후 shape: {features_safe.shape}')
print(f'\n정책 변수 첫 5행 (shift(1) 결과):')
print(features_safe[POLICY_VARS].head())

---

## 4. 타겟 Δy 생성 (bp 단위)

계획서 §2 — 타겟은 변화량 `Δy_t = (y_t − y_{t-1}) × 100   # 1-step ahead forecast (계획서 §2.1·§2.2)`.

본 노트북에서는 `Δy_t = (y_t − y_{t-1}) × 100` 를 **t 시점의 라벨**로 사용 (모델 학습 시 lag-feature 와 정렬 편의).

In [ ]:
y = features_safe[TARGET]
delta_y = y.diff() * 100  # bp
delta_y.name = 'delta_y_bp'

print(f'Δy 기초 통계 (bp):')
print(f'  N        = {delta_y.dropna().shape[0]:,d}')
print(f'  mean     = {delta_y.mean():+.3f}')
print(f'  std      = {delta_y.std():.3f}')
print(f'  min      = {delta_y.min():+.2f}')
print(f'  max      = {delta_y.max():+.2f}')
print(f'  q05      = {delta_y.quantile(0.05):+.2f}')
print(f'  q95      = {delta_y.quantile(0.95):+.2f}')
print(f'  |Δy|>5bp = {(delta_y.abs() > 5).mean()*100:.1f}%')
print(f'  |Δy|>10bp= {(delta_y.abs() > 10).mean()*100:.1f}%')

---

## 5. Lag Feature 생성 (CL-03)

- `pandas.shift(k)` (k ≥ 1) 만 사용 — 현재 시점 미포함 강제.
- config.yaml 의 `features.lags = [1, 5, 10, 20, 30]` 사용.

In [ ]:
LAGS = CONFIG['features']['lags']
print(f'Lag 시점: {LAGS}')

lag_blocks = []
for col in FREEZE_FEATURES:
    for k in LAGS:
        lag_blocks.append(features_safe[col].shift(k).rename(f'{col}__lag{k}'))

df_lag = pd.concat(lag_blocks, axis=1)
print(f'Lag feature shape: {df_lag.shape}')
print(f'생성 컬럼 예시: {df_lag.columns[:5].tolist()}')

---

## 6. Rolling Feature 생성 (CL-03)

- 모든 rolling 은 `.rolling(w).<agg>().shift(1)` 패턴 강제 — 현재 시점 미포함.
- 평균 / 표준편차 두 통계량 × 윈도우 [5, 10, 20] = 변수당 6개.

In [ ]:
ROLL_WINDOWS = [5, 10, 20]

roll_blocks = []
for col in FREEZE_FEATURES:
    for w in ROLL_WINDOWS:
        roll_blocks.append(features_safe[col].rolling(w).mean().shift(1).rename(f'{col}__rmean{w}'))
        roll_blocks.append(features_safe[col].rolling(w).std().shift(1).rename(f'{col}__rstd{w}'))

df_roll = pd.concat(roll_blocks, axis=1)
print(f'Rolling feature shape: {df_roll.shape}')
print(f'생성 컬럼 예시: {df_roll.columns[:6].tolist()}')

In [ ]:
# 통합 feature 행렬 + Δy 라벨
df_features = pd.concat([
    features_safe[FREEZE_FEATURES],   # raw (정책변수는 이미 shift 됨)
    df_lag,                            # lag
    df_roll,                           # rolling
    delta_y.to_frame()                 # 라벨
], axis=1)

# Lag/Rolling 으로 인한 초반 NaN drop
n_before = len(df_features)
df_features = df_features.dropna()
n_after = len(df_features)
print(f'NaN drop (초기 lag/rolling 워밍업): {n_before:,d} → {n_after:,d}  ({n_before - n_after}행 제거)')
print(f'최종 feature 수 (라벨 제외): {df_features.shape[1] - 1}')
print(f'기간: {df_features.index.min().date()} ~ {df_features.index.max().date()}')

---

## 7. Robust Scaler — Train-only fit (CL-02)

- `scaler.fit()` 은 **Train 구간에만** 호출.
- Cal/Val/Test 는 `transform()` 만.
- 이상치 영향 적은 RobustScaler 사용 (config.yaml `features.scaler = robust`).

In [ ]:
from sklearn.preprocessing import RobustScaler

FEATURE_COLS = [c for c in df_features.columns if c != 'delta_y_bp']

X_train_raw = slice_period(df_features, 'train')[FEATURE_COLS]
X_cal_raw   = slice_period(df_features, 'cal')[FEATURE_COLS]
X_val_raw   = slice_period(df_features, 'val')[FEATURE_COLS]
X_test_raw  = slice_period(df_features, 'test')[FEATURE_COLS]

scaler = RobustScaler()
scaler.fit(X_train_raw)  # ← Train-only fit (CL-02 핵심)

X_train = pd.DataFrame(scaler.transform(X_train_raw), index=X_train_raw.index, columns=FEATURE_COLS)
X_cal   = pd.DataFrame(scaler.transform(X_cal_raw),   index=X_cal_raw.index,   columns=FEATURE_COLS)
X_val   = pd.DataFrame(scaler.transform(X_val_raw),   index=X_val_raw.index,   columns=FEATURE_COLS)
X_test  = pd.DataFrame(scaler.transform(X_test_raw),  index=X_test_raw.index,  columns=FEATURE_COLS)

y_train = slice_period(df_features, 'train')['delta_y_bp']
y_cal   = slice_period(df_features, 'cal')['delta_y_bp']
y_val   = slice_period(df_features, 'val')['delta_y_bp']
y_test  = slice_period(df_features, 'test')['delta_y_bp']

print(f"{'split':10s}{'X.shape':>15s}{'y.shape':>15s}")
for name, X, yy in [('train',X_train,y_train),('cal',X_cal,y_cal),('val',X_val,y_val),('test',X_test,y_test)]:
    print(f"{name:10s}{str(X.shape):>15s}{str(yy.shape):>15s}")

# scaler 통계량이 train 만 반영했는지 sanity check
scaler_center_sample = pd.Series(scaler.center_[:5], index=FEATURE_COLS[:5])
train_median_sample  = X_train_raw.iloc[:, :5].median()
print(f'\nCL-02 검증 — scaler.center_ ≈ train.median() ?')
print(pd.concat([scaler_center_sample.rename('scaler.center_'),
                 train_median_sample.rename('train.median()')], axis=1).round(4))

---

## 8. Naive 베이스라인 (Δŷ = 0)

계획서 §5.1 — 일별 채권 변화량의 가장 강력한 무지 베이스라인.

- **점추정 RMSE/MAE**: 변화량 절댓값의 단순 평균 통계와 동일.
- **방향성 정확도**: Δŷ=0 은 sign 미정 → 50% 가정 시 동전 던지기 수준.
  - 실제 운용에서는 "전일 Δy 와 같은 부호" (모멘텀 Naive) 등이 더 강함. 본 분석에서는 단순 0 만.

In [ ]:
def metrics(y_true, y_pred, name):
    err = y_true - y_pred
    rmse = float(np.sqrt(np.mean(err ** 2)))
    mae  = float(np.mean(np.abs(err)))
    # 방향성 — 0 예측은 sign(0)=0 이라 일치 카운트 0. tie 는 제외하고 계산.
    mask = (np.sign(y_pred) != 0) & (np.sign(y_true) != 0)
    if mask.sum() > 0:
        dir_acc = float((np.sign(y_pred[mask]) == np.sign(y_true[mask])).mean())
    else:
        dir_acc = float('nan')
    return {'model': name, 'split': None, 'RMSE_bp': rmse, 'MAE_bp': mae, 'Dir_Acc': dir_acc}

naive_rows = []
for split_name, yy in [('train', y_train), ('cal', y_cal), ('val', y_val), ('test', y_test)]:
    pred = pd.Series(0.0, index=yy.index)
    row = metrics(yy, pred, 'Naive (Δŷ=0)')
    row['split'] = split_name
    naive_rows.append(row)

naive_df = pd.DataFrame(naive_rows)
print('=== Naive 베이스라인 (Δŷ=0) ===')
print(naive_df.to_string(index=False))

---

## 8.1 추가 방향성 베이스라인 — Momentum + Random (W2 보강)

계획서 §7 LSTM **방향성 정확도 55% 목표**의 비교 기준 명시.
Naive(Δŷ=0) 은 `sign(0)=0` 이라 dir_acc 측정 불가 → 두 추가 baseline 으로 보완:

| baseline | 정의 | 의미 |
|---|---|---|
| **Random (50%)** | 동전던지기 (analytical) | 통계적 최저 기준 |
| **Momentum** | `Δŷ_t = Δy_{t-1}` (전일 변화 지속) | 시장 모멘텀 가설, 운용 baseline |

> 코드 리뷰 W2 반영 (2026-05-02)

In [ ]:
# 추가 방향성 베이스라인 — Momentum + Random (LSTM 방향성 55% 비교 기준)

# (a) Momentum: 전체 timeline 에서 Δy.shift(1) 후 split 별 reindex
#     split 경계에서도 직전 영업일이 known 이라 누수 없음
y_full = pd.concat([y_train, y_cal, y_val, y_test]).sort_index()
momentum_full = y_full.shift(1)

momentum_rows = []
for split_name, yy in [('train', y_train), ('cal', y_cal), ('val', y_val), ('test', y_test)]:
    pred = momentum_full.reindex(yy.index)
    valid = ~pred.isna()
    row = metrics(yy[valid], pred[valid], 'Momentum (Δŷ=Δy_lag1)')
    row['split'] = split_name
    momentum_rows.append(row)
momentum_df = pd.DataFrame(momentum_rows)
print('=== Momentum 베이스라인 (Δŷ_t = Δy_{t-1}) ===')
print(momentum_df.to_string(index=False))

# (b) Random: 50% analytical 기준 (RMSE/MAE 는 의미 없음)
random_rows = []
for split_name, yy in [('train', y_train), ('cal', y_cal), ('val', y_val), ('test', y_test)]:
    n_nonzero = int((yy.dropna() != 0).sum())
    se = (0.5 * 0.5 / max(n_nonzero, 1)) ** 0.5
    random_rows.append({
        'model': 'Random (50%)', 'split': split_name,
        'RMSE_bp': float('nan'), 'MAE_bp': float('nan'),
        'Dir_Acc': 0.5,
    })
    print(f'  {split_name:6s} n={n_nonzero:>5d}  Random dir_acc=0.500 ± {1.96*se:.3f} (95% CI)')
random_df = pd.DataFrame(random_rows)

print()
print('📌 LSTM 목표 (계획서 §7): 방향성 정확도 ≥ 55%')
print('   - Random 50% 대비: +5%p 개선의 통계적 유의성 → 4주차 binomial test')
print('   - Momentum 대비: 시장 모멘텀(전일 변화 지속) 가설 보다 우수해야 채권 차별화')

---

## 9. ARIMA 베이스라인 (자기상관 학습 능력)

- 학습: Train 구간 Δy 만 사용 (params 고정).
- 평가: Cal/Val/Test 에 `apply(refit=False)` 로 state 만 확장 → one-step-ahead 예측.
  - 파라미터는 train 에서 한 번만 학습 → 미래 정보 누수 없음.
- 차수: ARIMA(1,0,1) on Δy (= ARIMA(1,1,1) on level).

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

ORDER = (1, 0, 1)

y_full = pd.concat([y_train, y_cal, y_val, y_test]).sort_index()

# (1) Train 만으로 적합
fit_train = ARIMA(y_train, order=ORDER, trend='c').fit()
print(f'ARIMA{ORDER} 학습 완료 — params:')
print(fit_train.params.round(5).to_string())
print(f'AIC = {fit_train.aic:.1f}')

# (2) state 확장 (params 고정, refit=False)
fit_full = fit_train.apply(y_full, refit=False)

# (3) one-step-ahead in-sample 예측 (state 전체로부터)
preds_full = fit_full.predict()

arima_rows = []
for split_name, yy in [('train', y_train), ('cal', y_cal), ('val', y_val), ('test', y_test)]:
    pred = preds_full.reindex(yy.index)
    row = metrics(yy.dropna(), pred.reindex(yy.dropna().index), f'ARIMA{ORDER}')
    row['split'] = split_name
    arima_rows.append(row)

arima_df = pd.DataFrame(arima_rows)
print('\n=== ARIMA 베이스라인 ===')
print(arima_df.to_string(index=False))

In [ ]:
# 비교 표 — 4개 베이스라인 (Naive, Momentum, Random, ARIMA)
baseline_compare = pd.concat([naive_df, momentum_df, random_df, arima_df], ignore_index=True)
baseline_compare = baseline_compare[['model', 'split', 'RMSE_bp', 'MAE_bp', 'Dir_Acc']]
baseline_compare[['RMSE_bp','MAE_bp']] = baseline_compare[['RMSE_bp','MAE_bp']].round(3)
baseline_compare['Dir_Acc'] = baseline_compare['Dir_Acc'].round(3)
print('=== 베이스라인 비교 (lower RMSE/MAE 더 좋음, Dir_Acc 1.0 이 최선) ===\n')
print(baseline_compare.to_string(index=False))

# Val/Test 방향성 정확도 요약 (LSTM 55% 비교 기준)
print('\n=== 방향성 정확도 비교 (LSTM 목표 ≥55% 대비) ===')
dir_pivot = baseline_compare[baseline_compare['split'].isin(['val','test'])].pivot(
    index='model', columns='split', values='Dir_Acc'
)
print(dir_pivot.to_string())

# 시각화 — Val/Test 의 RMSE 비교 (Random 은 NaN 이라 자동 제외)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
model_colors = {
    'Naive (Δŷ=0)':           'steelblue',
    'Momentum (Δŷ=Δy_lag1)':  'seagreen',
    'ARIMA(1, 0, 1)':         'darkorange',
}
for ax, sp in zip(axes, ['val', 'test']):
    sub = baseline_compare[(baseline_compare['split']==sp) & baseline_compare['RMSE_bp'].notna()]
    bar_colors = [model_colors.get(m, 'gray') for m in sub['model']]
    ax.bar(sub['model'], sub['RMSE_bp'], color=bar_colors, alpha=0.8)
    ax.set_title(f'{sp.upper()} — RMSE (bp)')
    ax.set_ylabel('RMSE (bp)')
    for i, v in enumerate(sub['RMSE_bp'].values):
        ax.text(i, v, f'{v:.2f}', ha='center', va='bottom')
    ax.tick_params(axis='x', rotation=15)
    ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(FIG_DIR / 'w2_04_baseline_compare.png', dpi=120, bbox_inches='tight')
plt.show()

---

## 10. 누수 차단 체크리스트 자동 검증 (CL-01 ~ CL-07)

> **2주차 후반 분리** — 본 셀의 인라인 grep 로직에서 false positive 4건이 발생해
> (마크다운 셀 매치, 자기 코드의 주석/정규식 패턴 매치, 정책금리 동일값으로 인한
> 검증 실패), `scripts/04_leakage_audit.py` 로 분리하고 다음을 강화:
>
> 1. `.ipynb` 를 JSON 파싱해 **code 셀만** 스캔 (마크다운 제외)
> 2. `#` 주석 줄 + `grep_repo(` 메타 코드 줄 + 감사 스크립트 자기 자신 제외
> 3. CL-05 는 `features_v1_candidate.csv` vs `features_with_lags_v1.csv` 의
>    정책 변수 컬럼 **직접 비교** (shift(1) 결과 일치 검증)
>
> 본 셀은 새 스크립트를 호출하고 결과만 표시한다.

In [ ]:
# 누수 체크리스트 — scripts/04_leakage_audit.py 호출 (강화 버전)
import subprocess, sys

audit_script = PROJECT_ROOT / 'scripts' / '04_leakage_audit.py'
result = subprocess.run(
    [sys.executable, str(audit_script)],
    capture_output=True, text=True, encoding='utf-8'
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    msg = f'\n🔴 감사 실패 (returncode={result.returncode}) — 위 ❌ 항목 수정 후 재실행'
    print(msg)
else:
    print('\n🟢 통과 — 2주차 누수 코드 리뷰 완료')

# 후속 셀 호환을 위해 audit_df 재로드
audit_df = pd.read_csv(REPORT_DIR / 'leakage_audit_w2.csv')

---

## 11. 리포트 템플릿 골격 작성

계획서 §8 — 최종 산출물 "성능 비교 리포트" + "오류 분석 리포트" 의 골격을 2주차에 미리 작성. 매주 산출물이 채워질 위치를 잡아둔다.

저장 위치: `reports/report_skeleton.md`

In [ ]:
skeleton_path = REPORT_DIR / 'report_skeleton.md'
skel = []
skel.append('# 채권금리 예측 — 최종 성능 비교 리포트 (골격)')
skel.append('')
skel.append('> 계획서 v5.1 §8 — 2주차 골격, 매주 채워나감.')
skel.append('> 생성: notebooks/02b_preprocess_baseline.ipynb (2주차)')
skel.append('')
skel.append('## 1. 요약 (Executive Summary)')
skel.append('- 1~2 문장 결론 — 7주차에 작성.')
skel.append('- 핵심 수치: Test 방향성 정확도, Pinball, Coverage, DM test p-value.')
skel.append('')
skel.append('## 2. 데이터')
skel.append('- 기간: 2010-01 ~ 2025-12 (영업일 기준)')
skel.append('- 분할: Train 2010-2020 / Cal 2021 / Val 2022 / Test 2023-2025')
skel.append('- 변수: 1주차 freeze 9개 + (3주차 ablation 후 최종 N개)')
skel.append('- 누수 차단: CL-01~CL-07 통과 (본 2주차 산출).')
skel.append('')
skel.append('## 3. 변수 선정 결과')
skel.append('- 1주차 EDA + 1주차 XGBoost-SHAP Hello World')
skel.append('- 2주차 상관·VIF·Granger (`docs/feature_validation_w2.md`)')
skel.append('- 3주차 freeze 최종 + 5주차 환율 ablation')
skel.append('')
skel.append('## 4. 베이스라인 — Naive · ARIMA (본 2주차)')
skel.append('')
skel.append('| 모델 | 분할 | RMSE (bp) | MAE (bp) | 방향성 |')
skel.append('|------|------|-----------|----------|--------|')
for _, r in baseline_compare.iterrows():
    skel.append(f"| {r['model']} | {r['split']} | {r['RMSE_bp']} | {r['MAE_bp']} | {r['Dir_Acc']} |")
skel.append('')
skel.append('> Figure: `reports/figures/w2_04_baseline_compare.png`')
skel.append('')
skel.append('## 5. XGBoost 분위수 회귀 (3주차 채움)')
skel.append('- `objective="reg:quantileerror"` 분위수 [0.05, 0.5, 0.95]')
skel.append('- Pinball Loss / Coverage / Sharpness 표')
skel.append('- Naive·ARIMA 대비 DM test 결과')
skel.append('')
skel.append('## 6. LSTM 분위수 회귀 (4-5주차 채움)')
skel.append('- 구조: 다변량 LSTM, lookback 30, hidden 64, layer 2, dropout 0.3')
skel.append('- 손실: Pinball Loss, monotonicity sort 후처리')
skel.append('- 학습 곡선·early stop')
skel.append('- (선택) Conformal CQR 후처리 — Coverage 미달 시')
skel.append('')
skel.append('## 7. 평가 지표 종합 (Naive · ARIMA · XGBoost · LSTM)')
skel.append('| 모델 | Pinball | RMSE | MAE | 방향성 | Coverage 90% | Sharpness | DM vs Naive |')
skel.append('|------|---------|------|-----|--------|--------------|-----------|-------------|')
skel.append('| Naive | … | … | … | 50%* | n/a | n/a | — |')
skel.append('| ARIMA | … | … | … | … | n/a | n/a | … |')
skel.append('| XGBoost | … | … | … | … | … | … | … |')
skel.append('| LSTM | … | … | … | … | … | … | … |')
skel.append('| LSTM+CQR | … | … | … | … | … | … | … |')
skel.append('')
skel.append('## 8. DM test (HLN 보정 + Bonferroni)')
skel.append('- Pinball Loss 차이 검정')
skel.append('- HAC(Newey-West) lag = …')
skel.append('- Bonferroni 다중비교 보정')
skel.append('')
skel.append('## 9. SHAP 분석 (6주차 채움)')
skel.append('- §6.3 핵심 분석 질문 5개 답변')
skel.append('- 분위수별 SHAP 차분 시각화')
skel.append('- 시차 효과 정량화 (lag k 별 |SHAP| 평균)')
skel.append('')
skel.append('## 10. 오류 분석 4축 (6주차 채움)')
skel.append('- (a) 방향성 오답 top-20')
skel.append('- (b) 큰 변동(|Δy|>5bp) 미예측 top-20')
skel.append('- (c) Coverage Miss — 위기 vs 정상, 보정 전후')
skel.append('- (d) 위기 vs 정상 구간 SHAP 평균 차이')
skel.append('')
skel.append('## 11. 환율 Ablation (5주차 채움)')
skel.append('- 환율 포함 모델 vs 미포함')
skel.append('- Pinball / Coverage / Sharpness / RMSE 비교')
skel.append('- 정부 개입 시점(예: 2024 계엄, 2025 고환율) SHAP 분석')
skel.append('')
skel.append('## 12. 위기구간 평가 (계획서 §4.4)')
skel.append('- 라벨링: 20일 rolling vol 상위 20% ∪ 이벤트 더미 ±1일')
skel.append('- 위기 vs 정상 RMSE 비율')
skel.append('- Coverage 위기 vs 정상')
skel.append('')
skel.append('## 13. 결론 및 한계')
skel.append('- 차별화 포인트 10개 중 어느 것이 결과로 입증되었는가')
skel.append('- 미달 항목의 솔직한 한계 + 후속 연구 방향')
skel.append('')
skel.append('## 부록 A. 누수 차단 체크리스트 결과')
skel.append('- CL-01 ~ CL-07 자동 검증 — `02b_preprocess_baseline.ipynb` §10')
skel.append('')
skel.append('## 부록 B. AI 사용 기록')
skel.append('- `AI_USAGE_LOG.md` — 매주 +2건, 총 14건+')
skel.append('- `VALIDATION_LOG.md` — 검증 사례')
skel.append('')

with open(skeleton_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(skel))
print(f'💾 저장: {skeleton_path.relative_to(PROJECT_ROOT)}')
print(f'   섹션: 13 + 부록 2  /  라인: {len(skel)}')

---

## 12. 산출물 저장

In [ ]:
# (1) Lag/Rolling 적용 + 라벨 — scaled 전 (3주차 XGBoost 학습 시 자유 변환)
out_features = DATA_DIR / 'processed' / 'features_with_lags_v1.csv'
df_features.to_csv(out_features, index_label='date')
print(f'💾 features_with_lags_v1.csv  shape={df_features.shape}  -> {out_features.relative_to(PROJECT_ROOT)}')

# (2) Scaler 학습 산물 — 3주차/4주차 재사용
import pickle
scaler_path = PROJECT_ROOT / 'models' / 'scaler_robust_train.pkl'
scaler_path.parent.mkdir(parents=True, exist_ok=True)
with open(scaler_path, 'wb') as f:
    pickle.dump({'scaler': scaler, 'feature_cols': FEATURE_COLS, 'split': SPLIT}, f)
print(f'💾 scaler_robust_train.pkl     -> {scaler_path.relative_to(PROJECT_ROOT)}')

# (3) 베이스라인 결과
out_baseline = REPORT_DIR / 'baseline_results_w2.csv'
baseline_compare.to_csv(out_baseline, index=False)
print(f'💾 baseline_results_w2.csv    -> {out_baseline.relative_to(PROJECT_ROOT)}')

# (4) 누수 검증 결과
out_audit = REPORT_DIR / 'leakage_audit_w2.csv'
audit_df.to_csv(out_audit, index=False)
print(f'💾 leakage_audit_w2.csv       -> {out_audit.relative_to(PROJECT_ROOT)}')

---

## 13. 다음 단계 (3주차)

다음 노트북 (`03_freeze_xgboost.ipynb`) 에서 수행:

1. **🔴 변수 freeze 최종 확정** (금) — 8개 + 사유 문서화
2. **5주차 ablation 대상 사전 명시** — 환율 1개 필수 + 추가 후보
3. **XGBoost 분위수 회귀** — `objective="reg:quantileerror"`, 분위수 [0.05, 0.5, 0.95]
4. **Naive·ARIMA·XGBoost 베이스라인 3개 비교표** (3주차 검증 포인트)

### 본 노트북 산출물 → 3주차 입력

- `data/processed/features_with_lags_v1.csv`
- `models/scaler_robust_train.pkl`
- `reports/baseline_results_w2.csv`
- `reports/leakage_audit_w2.csv`
- `reports/report_skeleton.md`

### TODO (이번 주 마감)

- [x] §10 누수 체크리스트 결과 → `VALIDATION_LOG.md` **#30** 에 통합 기록 (감사 스크립트 강화 + false positive 4건 분석)
- [ ] §8~§9 베이스라인 RMSE → `VALIDATION_LOG.md` **#33** 기록 (#31=VIF, #32=Granger 예약)
- [ ] §11 리포트 골격 → 팀원 공유 + 매주 채울 담당자 분담